# GPU：复用 CPU 缓存训练
选择 L4/A100 GPU 运行时，依次执行。先完成 CPU notebook，再运行这里。
原始数据路径、数据版本、滤波和采样参数必须与预热阶段一致。
`--require-cache` 禁止补算原始信号；缺失缓存会立即报错。
`--resume auto` 恢复同一目录的 last.ckpt；总轮数仍是 EPOCHS，不会额外再加 EPOCHS。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
REPO_URL = "https://github.com/wdongdong0708-jpg/brain2qwertyv1Test.git"
CODE = Path("/content/code")
WORKSPACE = Path("/content/drive/MyDrive/Brain2Qwerty_Workspace")
if (CODE / ".git").exists():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(CODE)], check=True)
os.chdir(CODE)
for folder in ("data", "cache_persisted", "archives", "runs"):
    (WORKSPACE / folder).mkdir(parents=True, exist_ok=True)
raw = WORKSPACE / "data/SpanishBCBL_Full"
if not raw.is_dir():
    raise FileNotFoundError(f"Put the already downloaded full SpanishBCBL repository at {raw}")
for name in ("Pinet2024Eeg", "Pinet2024Meg"):
    alias = WORKSPACE / "data" / name
    if not alias.exists():
        try:
            alias.symlink_to(raw, target_is_directory=True)
        except OSError:
            print(f"Drive does not support this symlink; the reader will resolve {raw} directly")
os.environ["BRAIN2QWERTY_STUDIES"] = str(WORKSPACE / "data")
os.environ["BRAIN2QWERTY_RESULTS"] = str(WORKSPACE / "runs")
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "scripts.colab_setup"], check=True)
PYTHON = "/content/b2q-venv/bin/python"


In [ ]:
os.environ["BRAIN2QWERTY_CACHE"] = "/content/local_cache"
subprocess.run([PYTHON, "-m", "scripts.cache_archive", "unpack",
                "--archive", str(WORKSPACE / "archives/features.tar"),
                "--cache-root", os.environ["BRAIN2QWERTY_CACHE"]], check=True)


In [ ]:
MODALITY = "eeg"   # eeg / meg
SUBJECT = "all"    # 002 for one EEG subject; S2 for one MEG subject
EPOCHS = 300       # finite total budget; resume must keep the original value
SEED = 33
subprocess.run([PYTHON, "-u", "-m", "brain2qwerty_v1", "train",
                "--modality", MODALITY, "--subject", SUBJECT,
                "--seed", str(SEED), "--epochs", str(EPOCHS),
                "--devices", "1", "--require-cache", "--resume", "auto"], check=True)


Checkpoint、逐步日志、每轮状态均直接写入 Drive 的 runs/。同一实验再次执行会恢复断点；模型结构、batch、种子和总轮数需保持一致。新实验请指定不同的 `--run-dir`。
